# Corrupted Data Evaluation
Systematische Evaluation des Ensembles über verschiedene Corruption-Typen und Severity-Level

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repository
!git clone https://github.com/deadPixelsGreta/xAI-proj-m-ws2526.git

In [ ]:
# Setup project root
import sys, os
from pathlib import Path

def find_project_root(start: Path) -> Path:
    markers = {".git", "requirements.txt", "setup.py", "pyproject.toml"}
    root = None
    for parent in [start, *start.parents]:
        if any((parent / m).exists() for m in markers):
            root = parent
    return root or start

cloned_repo_name = "xAI-proj-m-ws2526"
cloned_repo_path = Path.cwd() / cloned_repo_name

if cloned_repo_path.is_dir():
    os.chdir(cloned_repo_path)

ROOT = find_project_root(Path.cwd()).resolve()
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

print("cwd:", Path.cwd())
print("root on sys.path:", str(ROOT) in sys.path)

In [ ]:
# Install dependencies
!pip install -r experiments/base_ensemble/requirements.txt --quiet

## Load Corrupted Dataset

In [ ]:
# 1. Copy zip from Drive to local VM
!cp /content/drive/MyDrive/corrupted.zip /content/

# 2. Unzip directly to /content/
!unzip -q /content/corrupted.zip -d /content/

# 3. Move to datasets folder
!mv /content/corrupted /content/xAI-proj-m-ws2526/datasets/

# 4. Remove the zip to save space
!rm /content/corrupted.zip

## Systematische Evaluation über alle Corruption-Typen und Levels

In [ ]:
import subprocess
import re
from pathlib import Path
import pandas as pd

# Definieren Sie Ihre Checkpoints
CHECKPOINT_DIR = "/content/drive/MyDrive/best_models_resnet"
CHECKPOINT = "best_resnet50-20260130-181509_epoch8.pth"

# Corruption-Typen (Ordner im datasets/corrupted/)
corruption_types = ['gaussian_noise', 'pixelate']  # Erweitern Sie diese Liste nach Bedarf

# Severity levels
severity_levels = [1, 2, 3, 4, 5]

# Ergebnis-Speicher
results = []

print("="*80)
print("Starting Systematic Corruption Evaluation")
print("="*80)

for corruption in corruption_types:
    print(f"\n{'='*80}")
    print(f"Corruption Type: {corruption.upper()}")
    print("="*80)
    
    for severity in severity_levels:
        print(f"\n{'-'*80}")
        print(f"Evaluating: {corruption} - Severity Level {severity}")
        print("-"*80)
        
        # Pfad zum Severity-Level (inkl. test/)
        data_dir = f"datasets/test/{corruption}/severity_{severity}"  # ← test/ ist Teil des Pfads!
        
        # Überprüfen, ob der Pfad existiert
        if not Path(data_dir).exists():
            print(f"⚠️  Skipping: {data_dir} does not exist")
            continue
        
        # Führen Sie die Evaluation aus
        cmd = [
            "python", "-m", "experiments.base_ensemble.scripts.ensemble_inference",
            "--evaluate",
            "--data-dir", data_dir,
            "--split", "",  # ← Kein weiterer split, da test/ schon im Pfad ist!
            "--checkpoints", f"{CHECKPOINT_DIR}/{CHECKPOINT}"
        ]
        
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, check=True)
            output = result.stdout
            
            # Extrahieren Sie die Overall Accuracy
            accuracy_match = re.search(r'Overall Accuracy.*?(\d+\.\d+)%', output)
            if accuracy_match:
                accuracy = float(accuracy_match.group(1))
                results.append({
                    'Corruption': corruption,
                    'Severity': severity,
                    'Accuracy': accuracy
                })
                print(f"✓ Overall Accuracy: {accuracy:.2f}%")
            
            # Ausgabe der vollständigen Ergebnisse
            print("\n" + output)
            
        except subprocess.CalledProcessError as e:
            print(f"❌ Error during evaluation: {e}")
            print(e.stderr)

print("\n" + "="*80)
print("Evaluation Complete")
print("="*80)

## Zusammenfassung der Ergebnisse

In [ ]:
# Erstellen Sie eine übersichtliche Tabelle
df = pd.DataFrame(results)

if not df.empty:
    # Pivot-Tabelle für bessere Übersicht
    pivot_df = df.pivot(index='Severity', columns='Corruption', values='Accuracy')
    
    print("\n" + "="*80)
    print("SUMMARY: Accuracy across Corruption Types and Severity Levels")
    print("="*80)
    print(pivot_df.to_string())
    
    # Speichern Sie die Ergebnisse
    df.to_csv('corruption_evaluation_results.csv', index=False)
    print("\n✓ Results saved to: corruption_evaluation_results.csv")
    
    # Optional: Visualisierung
    import matplotlib.pyplot as plt
    
    plt.figure(figsize=(12, 6))
    for corruption in df['Corruption'].unique():
        subset = df[df['Corruption'] == corruption]
        plt.plot(subset['Severity'], subset['Accuracy'], marker='o', label=corruption)
    
    plt.xlabel('Severity Level')
    plt.ylabel('Accuracy (%)')
    plt.title('Model Robustness across Corruption Types')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks([1, 2, 3, 4, 5])
    plt.savefig('corruption_evaluation_plot.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Plot saved to: corruption_evaluation_plot.png")
else:
    print("⚠️  No results collected. Please check the evaluation output.")

## Optional: Speichern der Ergebnisse auf Google Drive

In [ ]:
# Kopieren Sie die Ergebnisse zu Google Drive
!cp corruption_evaluation_results.csv /content/drive/MyDrive/
!cp corruption_evaluation_plot.png /content/drive/MyDrive/

print("✓ Results copied to Google Drive")